## Projects in Data Science - Exercises for the Annotation lectures

In these exercises you will use some generated and exisiting (collected from human observer) data, to practice working with inter-observer agreement.

The existing data we are working with are annotations made for skin lesion images from the ISIC archive (https://www.isic-archive.com/). The annotations describe different characteristics of how the lesion look - for example, whether they are asymmetric or not.


You can download (group07.csv) the data here: https://github.com/raumannsr/ENHANCE/tree/main/0_data/student/2017-2018


Each row is a different image, and each column is a different feature done by a specific annotator. For example: Asymmetry_7_1 is the feature Asymmetry annotated by annotator 7_1. There are 100 lesions, each of which was annotated by six annotators in total, five different features, and six different annotators. The feature are either binary, or ordinal.   

These annotations were made in 2017 by students at TU Eindhoven. You can (optionally) read more about the background and how the annotations can be used in these papers:

* https://arxiv.org/pdf/1806.08174
* https://www.melba-journal.org/papers/2021:020.html



In [6]:
# Load skin lesion annotations as a dataframe
import pandas as pd
import numpy as np

df = pd.read_csv("../data/group07.csv", sep=";")
df.head()

,ID,Asymmetry_7_1,Color_7_1,Border_7_1,Dermo_7_1,Blue_7_1,Asymmetry_7_2,Color_7_2,Border_7_2,Dermo_7_2,...,Asymmetry_7_5,Color_7_5,Border_7_5,Dermo_7_5,Blue_7_5,Asymmetry_7_6,Color_7_6,Border_7_6,Dermo_7_6,Blue_7_6
0,ISIC_0000549,2,4,1,1,0,2,5,1,2,...,2,5,1,2,1,2.0,5.0,1.0,2.0,1.0
1,ISIC_0000550,1,3,1,1,0,2,4,1,1,...,1,5,1,1,0,1.0,4.0,1.0,1.0,0.0
2,ISIC_0000551,2,2,1,2,0,2,3,1,2,...,1,4,1,1,0,2.0,2.0,1.0,2.0,0.0
3,ISIC_0000552,1,4,1,1,1,2,4,1,1,...,2,4,1,1,0,1.0,3.0,1.0,1.0,0.0
4,ISIC_0000554,2,3,1,1,1,2,5,1,2,...,2,5,1,2,0,2.0,3.0,1.0,1.0,1.0


## How often do annotators agree?

Select Blue which is a binary feature, for two different annotators. Calculate how often they agree in percent. This is the same as the accuracy metric but you can do this without any imports.



In [7]:
## Select two annotators and compute observed agreement ("accuracy")

blue_cols = [c for c in df.columns if c.startswith("Blue_7_")]
ann1, ann2 = blue_cols[0], blue_cols[1]

pair = df[[ann1, ann2]].dropna()
observed_agreement = (pair[ann1] == pair[ann2]).mean()

print("Annotators:", ann1, "vs", ann2)
print("Observed agreement (%):", observed_agreement * 100)

Annotators: Blue_7_1 vs Blue_7_2
Observed agreement (%): 82.0


Make a 6x6 array where you loop through the different annotators, and calculate their agreement. You should see that for all annotator pairs, the percentage of agreement is between 80 and 90.


In [8]:
## Loop through the annotators

blue_cols = [c for c in df.columns if c.startswith("Blue_7_")]

agreement_matrix = np.zeros((len(blue_cols), len(blue_cols)))

for i, c1 in enumerate(blue_cols):
    for j, c2 in enumerate(blue_cols):
        pair = df[[c1, c2]].dropna()
        agreement_matrix[i, j] = np.mean(pair[c1].to_numpy() == pair[c2].to_numpy())

agreement_df = pd.DataFrame(agreement_matrix, index=blue_cols, columns=blue_cols)
agreement_df * 100

,Blue_7_1,Blue_7_2,Blue_7_3,Blue_7_4,Blue_7_5,Blue_7_6
Blue_7_1,100.000000,82.000000,91.836735,83.000000,82.000000,83.838384
Blue_7_2,82.000000,100.000000,82.653061,77.000000,78.000000,77.777778
Blue_7_3,91.836735,82.653061,100.000000,87.755102,83.673469,88.775510
Blue_7_4,83.000000,77.000000,87.755102,100.000000,87.000000,94.949495
Blue_7_5,82.000000,78.000000,83.673469,87.000000,100.000000,87.878788
Blue_7_6,83.838384,77.777778,88.775510,94.949495,87.878788,100.000000


## How can we interpret this agreement measure?

Look at how often each annotator found that the lesion has value 1 for the Blue feature.


In [9]:
## Find how often each annotator thought the lesion had the Blue feature

blue_cols = [c for c in df.columns if c.startswith("Blue_7_")]
blue_prevalence = df[blue_cols].mean()
blue_prevalence

Blue_7_1    0.200000
Blue_7_2    0.310000
Blue_7_3    0.173469
Blue_7_4    0.170000
Blue_7_5    0.160000
Blue_7_6    0.141414
dtype: float64

Create a "random" feature (like an annotator who did not look at the image at all), where the Blue feature occurs as often as in the real annotations. Then again calculate the agreement. Is the result what you would expect?

In [10]:
# Create random annotation with the same prevalence of Blue adnd calculate the observed agreement

blue_cols = [c for c in df.columns if c.startswith("Blue_7_")]
ann = blue_cols[0]

series = df[ann].dropna()
p_blue = series.mean()

np.random.seed(1234)
random_ann = np.random.binomial(1, p_blue, size=len(series))

random_agreement = np.mean(series.to_numpy() == random_ann)

print("Annotator:", ann)
print("Prevalence of Blue (1):", p_blue)
print("Agreement with random annotator (%):", random_agreement * 100)

Annotator: Blue_7_1
Prevalence of Blue (1): 0.2
Agreement with random annotator (%): 72.0


Select only the lesions where at least one annotator thought it had the Blue feature. Calculate the agreements again (as percentage out of 100. What do you notice?

In [11]:
# Find all lesions with at least one Blue annotation, and calculate agreement again (in %)

blue_cols = [c for c in df.columns if c.startswith("Blue_7_")]
blue_data = df[blue_cols]

subset = blue_data[(blue_data == 1).any(axis=1)]

agreement_subset = np.zeros((len(blue_cols), len(blue_cols)))

for i, c1 in enumerate(blue_cols):
    for j, c2 in enumerate(blue_cols):
        pair = subset[[c1, c2]].dropna()
        agreement_subset[i, j] = np.mean(pair[c1].to_numpy() == pair[c2].to_numpy())

agreement_subset_df = pd.DataFrame(agreement_subset, index=blue_cols, columns=blue_cols)
agreement_subset_df * 100


,Blue_7_1,Blue_7_2,Blue_7_3,Blue_7_4,Blue_7_5,Blue_7_6
Blue_7_1,100.000000,48.571429,76.470588,51.428571,48.571429,54.285714
Blue_7_2,48.571429,100.000000,50.000000,34.285714,37.142857,37.142857
Blue_7_3,76.470588,50.000000,100.000000,64.705882,52.941176,67.647059
Blue_7_4,51.428571,34.285714,64.705882,100.000000,62.857143,85.714286
Blue_7_5,48.571429,37.142857,52.941176,62.857143,100.000000,65.714286
Blue_7_6,54.285714,37.142857,67.647059,85.714286,65.714286,100.000000


## Cohen's Kappa

As you maybe see above, total % of agreement may not reflect what you want to find out about your annotations, if the different values are not occuring equally often.

Instead let's look at the Kappa score, which adjusts for this. You can calculate it yourself it from the observed agreement (on all lesions) above. See https://en.wikipedia.org/wiki/Cohen%27s_kappa or the Viera paper on LearnIT:

In [12]:
# Calculate kappa score "by hand" (example)

print(df[ann1].unique())
print(df[ann2].unique())

def cohen_kappa(a, b):
    data = pd.DataFrame({"a": a, "b": b}).dropna()
    labels = sorted(set(data["a"].unique()) | set(data["b"].unique()))
    conf = pd.crosstab(data["a"], data["b"]).reindex(
        index=labels, columns=labels, fill_value=0
    )
    n = conf.values.sum()
    po = np.trace(conf.values) / n
    row_marginals = conf.sum(axis=1).values / n
    col_marginals = conf.sum(axis=0).values / n
    pe = np.dot(row_marginals, col_marginals)
    kappa = (po - pe) / (1 - pe)
    return kappa, po, pe, conf

blue_cols = [c for c in df.columns if c.startswith("Blue_7_")]
ann1, ann2 = blue_cols[0], blue_cols[1]

kappa, po, pe, conf = cohen_kappa(df[ann1], df[ann2])

print("Confusion matrix:")
print(conf)
print("Observed agreement Po:", po)
print("Expected agreement Pe:", pe)
print("Cohen's kappa:", kappa)

[0 1]
[1 0 2]
Confusion matrix:
b   0   1  2
a           
0  72   7  1
1   4  10  6
2   0   0  0
Observed agreement Po: 0.82
Expected agreement Pe: 0.6420000000000001
Cohen's kappa: 0.4972067039106142


Check your answer against the Kappa score available in sklearn.metrics.

Note that this score is only for two annotators and categorical variables. For extensions of the Cohen's Kappa you will need to adapt your own function, or use other packages.

In [13]:
# Calculate Kappa score with sklearn

from sklearn.metrics import cohen_kappa_score

blue_cols = [c for c in df.columns if c.startswith("Blue_7_")]
ann1, ann2 = blue_cols[0], blue_cols[1]

pair = df[[ann1, ann2]].dropna()
kappa_sklearn = cohen_kappa_score(pair[ann1], pair[ann2])

print("Cohen's kappa:", kappa_sklearn)


Cohen's kappa: 0.4972067039106145


## How often do annotators agree?

For Blue_7_1 vs Blue_7_2 the raw agreement is 82% (they give the same label in 82 out of 100 lesions).
In all 6 Blue annotators, pairwise agreements range from about 77% to 95%, with an average off-diagonal agreement of ~84.5%.
The most similar pair overall is Blue_7_4 vs Blue_7_6 (~95% agreement) and the leas t similar is Blue_7_2 vs Blue_7_4 (~77% agreement.
So if we just look at “% equal labels”, the annotators appear to agree quite a lot.

---

## How can we interpret this agreement measure?

The Blue feature is rare: depending on the annotator, “Blue = 1” occurs only in ~14–31% of lesions (most labels are 0).
High raw agreement is in that case heavily driven by everyone agreeing on “no Blue” (0), not necessarily on the positive cases.
Because of this imbalance, a high % agreement overestimates how reliable/consistent the Blue annotation really is.
So 80–90% agreement sounds strong, but given the skewed label distribution it’s actually less impressive than it looks.

---

## Create a “random” feature (like an annotator who did not look at the image) and calculate the agreement. Is the result what you would expect?

A random annotator with the same prevalence as Blue_7_1 (p ~0.20) already reaches about 71% agreement with the real annotator.
This shows that we can get a surprisingly high % agreement purely from the label distribution.
The real annotator vs annotator agreement (82%) is only about 11 % points above this random baseline.

---

## Select only the lesions where at least one annotator thought it had value 1 for the Blue feature and calculate the agreements again. What do you notice?

Restricting to lesions where at least one annotator marked Blue = 1 leaves 35 lesions.
On this subset, pairwise agreements drop: off-diagonal values are now in the ~34–86% range, with an average around 56%.
The worst pair is again Blue_7_2 vs Blue_7_4 (~34% agreement), and the best is Blue_7_4 vs Blue_7_6 (~86%).
So on the `interesting` lesions (possible Blue), annotators disagree much ore.

---

## Cohen’s Kappa

For Blue_7_1 vs Blue_7_2 we get:
- Observed agreement Po = 0.82
- Expected agreement by chance Pe ~0.642
- Cohen’s k ~0.50

k~0.5 interprets as `moderate` agreement, not `excellent`.  
Kappa corrects for the high chance agreement caused by the imbalanced labels and shows that the true reliability is only moderate.  
The confusion matrix also shows that most cases are 0/0, while positive/positive and between-category (1 vs 2) agreements are much weaker.

---

## Check your answer against the Kappa score available in sklearn.metrics.

The sklearn `cohen_kappa_score` for Blue_7_1 vs Blue_7_2 is ~0.497, which matches the manually computed k.  
This confirms that the implementation of the hand-written kappa is correct.
Both methods agree that, after adjusting for chance, the Blue annotation shows only moderate reliability.